# 4.3 A GWAS with Hail (Python)

**Omics series, class 3 of 7.** In 4.2 we ran a GWAS of type 2 diabetes in the *TCF7L2* region with **PLINK**, from R. Today we run the **same analysis** with **Hail**, a Python library for genomic data, and compare the results. *All of Us* uses Hail for its own large analyses, and its genomic data comes ready to use in Hail's format.

This notebook is in **Python**. You don't need to know Python already: every step is explained, and there's a short guide for R users in Part 1.

### By the end of this notebook you'll be able to

1. Explain what Hail is, and how a Hail **MatrixTable** holds genomic data
2. Read part of the *All of Us* genomes with Hail, keep our 5,000 people, and save a small working copy
3. Run QC and a logistic-regression GWAS with Hail
4. Turn Hail results into a pandas table, and plot them
5. Compare Hail's results with PLINK's, and explain why they differ slightly

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | `gwas_samples.csv` (the sample, from **4.1**), PLINK's results file from **4.2**, and the v9 genomic files |
| 💾 **Saves** | `gwas_tcf7l2_hail.csv`: Hail's results |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: Hail, MatrixTables, Python for R users | 25 |
| 2. Setup and samples | 10 |
| 3. Read the genomes with Hail | 20 |
| 4. Quality control | 15 |
| 5. Run the GWAS | 15 |
| 6. Results and plots | 15 |
| 7. Compare with PLINK; scaling up; wrap-up | 20 |

**Before you start:**

- Run **4.1** and **4.2** first: this notebook uses their sample and PLINK results.
- Open this notebook with the **Python 3** kernel (top right of the notebook).
- `aou_setup.py` reads your bucket name from `aou_setup.R`, so there's nothing new to set up.

> 💰 **Cost:** runs on a **standard app** (no cluster). Reading the genomes in Part 3 takes about **2 to 3 minutes** (it reads data for all 535,000 people in a small window); after that, every step takes seconds. No BigQuery queries. Stop the app when you're done.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including genotypes and IDs), or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### What is Hail?

**Hail** is an open-source Python library for genomic data, built by the Broad Institute. It's designed for **very large** datasets: it splits the work into pieces and can run them on many computers at once, using a system called **Spark**. On a standard app, Spark uses the app's own processors.

| | PLINK (4.2) | Hail (4.3) |
|---|---|---|
| Runs from | the command line (we called it from R) | Python |
| Data format | `.pgen`/`.pvar`/`.psam` files | a **MatrixTable** (a folder, `.mt`) |
| Strengths | very fast for standard tests on one computer | flexible; scales to a cluster; *All of Us*'s own tool |
| Used by *All of Us* for | | quality control, the All by All results (4.4), many tutorials |

### The MatrixTable

A Hail **MatrixTable** is a big grid with extra information along each side:

| Part | In *All of Us* | Example fields |
|---|---|---|
| **Rows** | variants | `locus` (chromosome:position), `alleles` (`["C", "T"]`) |
| **Columns** | people | `s` (the person's ID: same as `person_id`) |
| **Entries** (row × column) | one person's genotype at one variant | `GT` (genotype, e.g. `0/1`) |
| **Globals** | facts about the whole dataset | |

You **annotate** (add fields to) rows, columns, or entries, and **filter** rows or columns, much like `mutate()` and `filter()` in R.

### Lazy evaluation

Hail is **lazy**: when you filter or annotate, it only **writes down the plan**. Nothing is read until you ask for an answer: a count, a table to look at, or a file to write. So the filtering cells below finish instantly, and the cell that asks for a count does all the work. If a cell suddenly takes minutes, it's because it triggered the plan.

### splitMT: one row per ALT allele

*All of Us* provides the ACAF genomes in two Hail versions:

| Folder | Variants with several ALT alleles (4.1 Part 6) |
|---|---|
| `multiMT` | one row, with all the alleles: `["C", "T", "G"]` |
| `splitMT` | **split** into one row per ALT allele: `["C", "T"]` and `["C", "G"]` |

We use **splitMT**, because tests like logistic regression need one ALT allele per row. So there are **more rows** than in PLINK's files: the repeat (STR) at the top of 4.2, with about 15 ALT alleles, becomes about 15 rows.

### Python for R users

| Idea | R | Python |
|---|---|---|
| Load a package | `library(dplyr)` | `import pandas as pd` (then `pd.read_csv()`) |
| Data frame | tibble / data frame | pandas **DataFrame** |
| Use a function on an object | `filter(df, ...)` or `df %>% filter(...)` | a **method**: `df.query(...)`, `mt.filter_rows(...)` |
| Column | `df$age` | `df["age"]` or `df.age` |
| Text with values in it | `str_glue("{x} people")` | `f"{x} people"` (an **f-string**) |
| Logical values | `TRUE`, `FALSE` | `True`, `False` |
| Counting from | 1 | **0**: the first item of a list is `x[0]` |
| Lists | `c(1, 2, 3)` | `[1, 2, 3]` |
| Missing value | `NA` | `NaN` or `None` |

### Glossary

| Term | Means |
|---|---|
| **Spark** | the system Hail uses to split work into pieces and run them in parallel |
| **MatrixTable** | Hail's data structure: rows (variants) × columns (people), with entries (genotypes) |
| **Locus** | a position in the genome: chromosome and position |
| **Checkpoint** | save an intermediate result to disk, so later steps read the small copy instead of redoing the work |
| **Wald test** | the standard test in logistic regression; what PLINK uses when it doesn't need Firth |

## Part 2. Setup and samples

`sys.path.insert(0, "..")` tells Python to look for `aou_setup.py` in the folder above; `from aou_setup import *` loads its settings (`DATA_DIR`) and helpers (`aou_data_dir()`, `hide_small_counts()`).

`hl.init()` starts Hail (and Spark). It prints a lot, including warnings. Two are expected and harmless here:

- *Hail was built and tested with Java 11. You are using Java 17* : the app's Java version; Hail works with it
- *This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.9* : a minor version difference

`log=` writes Hail's log file outside the repository folder, so it doesn't clutter it.

In [ ]:
import sys
sys.path.insert(0, "..")
from aou_setup import *          # DATA_DIR, aou_data_dir(), hide_small_counts()

import glob
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hail as hl

os.makedirs(os.path.expanduser("~/hail_logs"), exist_ok=True)
hl.init(log=os.path.expanduser("~/hail_logs/hail.log"), idempotent=True)
hl.default_reference("GRCh38")

AOU_DIR  = aou_data_dir("v9")
MT_PATH  = os.path.join(AOU_DIR, "wgs", "short_read", "snpindel", "acaf_threshold", "splitMT", "hail.mt")

# A 200,000-letter window over the TCF7L2 signal from 4.2 (GRCh38)
WINDOW     = "chr10:112900000-113100000"
GENE_START = 112950247   # TCF7L2
GENE_END   = 113167678
LEAD_POS   = 112998590   # rs7903146

### The sample from 4.1 and 4.2

We read `gwas_samples.csv` with **pandas**. `dtype={"person_id": str}` reads the IDs as **text**, to match the genome IDs, just as in R.

Then we make the columns the model needs:

| Column | Means |
|---|---|
| `is_case` | `True` for type 2 diabetes cases, `False` for controls |
| `sex_male` | 1 = male, 0 = female |
| `age`, `PC1` … `PC10` | as in 4.2 |

In [ ]:
samples = pd.read_csv(os.path.join(DATA_DIR, "gwas_samples.csv"), dtype={"person_id": str})

samples["is_case"]  = samples["cohort"] == "case"
samples["sex_male"] = (samples["sex_at_birth"] == "Male").astype(int)

PCS = [f"PC{i}" for i in range(1, 11)]   # ["PC1", "PC2", ..., "PC10"]
samples = samples[["person_id", "cohort", "is_case", "age", "sex_male"] + PCS]

# People without PCs can't be adjusted for ancestry: leave them out, as PLINK did in 4.2
print("Missing PCs:", samples[PCS].isna().any(axis=1).sum())
samples = samples.dropna(subset=PCS)

samples["cohort"].value_counts()

**🔍 Read the output:** the number of people missing PCs (don't share it if it's 1 to 20), then about 2,500 cases and 2,500 controls: the same people as in 4.2.

Hail can't use a pandas DataFrame directly, so we turn it into a **Hail Table**, **keyed** by `person_id` (the key is what Hail matches on, like `by =` in a join).

In [ ]:
samples_ht = hl.Table.from_pandas(samples, key="person_id")
samples_ht.describe()

**🔍 Read the output:** `describe()` lists the table's fields and their **types**: `str` (text), `bool` (True/False), `int`, `float`. `Key: ['person_id']` is the field Hail joins on.

## Part 3. Read the genomes with Hail

### Open the MatrixTable

`hl.read_matrix_table()` **opens** the MatrixTable: thanks to lazy evaluation, it reads only the description, not the genotypes. `describe()` shows the structure.

In [ ]:
mt = hl.read_matrix_table(MT_PATH)
mt.describe()

**🔍 Read the output:** look for the four parts from Part 1:

- **Column fields:** `s`, the person's ID (`str`)
- **Row fields:** `locus` (`locus<GRCh38>`), `alleles` (`array<str>`), and more
- **Entry fields:** `GT` (`call`, the genotype), and possibly others
- **Keys:** rows are keyed by `locus` and `alleles`, columns by `s`

### Keep our window and our people

| Step | Hail | Like, in R |
|---|---|---|
| keep only variants in the window | `hl.filter_intervals(mt, [...])` | PLINK's `--chr --from-bp --to-bp` |
| keep only our 5,000 people | `mt.semi_join_cols(samples_ht)` | `semi_join()`, or PLINK's `--keep` |
| add the phenotype and covariates to each person | `mt.annotate_cols(pheno = samples_ht[mt.s])` | `left_join()` |

`samples_ht[mt.s]` means "the row of `samples_ht` whose key matches this person's `s`".

These lines are **instant**: they only add to the plan.

In [ ]:
mt = hl.filter_intervals(mt, [hl.parse_locus_interval(WINDOW)])
mt = mt.semi_join_cols(samples_ht)
mt = mt.annotate_cols(pheno = samples_ht[mt.s])

### Save a small working copy (checkpoint)

Now the slow step. `checkpoint()` runs the plan: it reads the window for **all 535,000 people** from the bucket (the files store everyone's genotypes together), keeps our 5,000, and **writes** the result to the app's disk. That takes about **2 to 3 minutes**. Every later step reads this small copy instead, in seconds.

The copy is participant-level data, so it stays on the app's disk in your home folder (`~/hail_checkpoints/`), inside the Workbench. If the copy already exists (e.g. you're rerunning the notebook), we just open it. Set `REBUILD = True` if you change the window or the sample.

While it runs, a progress bar like `[Stage 1:=====>   (3 + 1) / 7]` shows Spark working through the pieces.

In [ ]:
REBUILD         = False
CHECKPOINT_PATH = os.path.expanduser("~/hail_checkpoints/tcf7l2_window.mt")

start = time.time()
if REBUILD or not os.path.exists(CHECKPOINT_PATH):
    mt = mt.checkpoint(CHECKPOINT_PATH, overwrite=True)
else:
    mt = hl.read_matrix_table(CHECKPOINT_PATH)
print(f"{time.time() - start:.0f} seconds")

n_variants, n_people = mt.count()
print(f"{n_variants:,} variants (rows) x {n_people:,} people (columns)")

**🔍 Read the output:** about 2 to 3 minutes the first time (150 seconds in our test run), then 5,000 people and about **12,000 rows**. That's more than PLINK would have for this window, because `splitMT` has one row per ALT allele.

`mt.aggregate_cols()` summarises the columns. `hl.agg.counter()` counts each value, like `count()` in R. (Hail may print a warning that it *aggregates over cols ordered by 'col_key'*: that's about the order of people, which doesn't matter for a count.)

In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.pheno.cohort))

### ✍️ Your turn

The repeat (STR) that topped the 4.2 results is at position **113,036,665**. How many rows does it have in this MatrixTable? Use `mt.filter_rows()` and `count_rows()`.

<details>
<summary><b>Show answer</b></summary>

```python
mt.filter_rows(mt.locus.position == 113036665).count_rows()
```

One row per ALT allele (about 15). In PLINK's files it was **one** row with all the alleles. To see the alleles:

```python
mt.filter_rows(mt.locus.position == 113036665).alleles.collect()
```

`.collect()` brings the values into Python as a list. It's fine for a few values; never collect genotypes.

</details>

In [ ]:
# Your code here

## Part 4. Quality control

The same filters as in 4.2, written the Hail way. `hl.variant_qc()` calculates QC statistics for every row and stores them in a row field called `variant_qc`. (`describe()` showed a `variant_qc` field already: *All of Us* calculated it for **all** 535,000 people. We recalculate it for **our** 5,000, replacing it.)

| Field | Means | Filter (same as 4.2) |
|---|---|---|
| `variant_qc.AF` | the frequency of each allele, e.g. `[0.72, 0.28]` | the **smaller** one (`hl.min()`) above 0.01 |
| `variant_qc.call_rate` | the share of people with a genotype | above 0.9 |
| `variant_qc.p_value_hwe` | the Hardy-Weinberg test p-value | above 10⁻¹⁰ |

First we count how many rows **fail** each filter, then keep the rows that pass all three.

In [ ]:
mt = hl.variant_qc(mt)

passes_maf  = hl.min(mt.variant_qc.AF) > 0.01
passes_call = mt.variant_qc.call_rate > 0.9
passes_hwe  = mt.variant_qc.p_value_hwe > 1e-10

mt.aggregate_rows(hl.struct(
    fail_maf  = hl.agg.count_where(~passes_maf),
    fail_call = hl.agg.count_where(~passes_call),
    fail_hwe  = hl.agg.count_where(~passes_hwe),
))

**🔍 Read the output:** most rows fail the **frequency** filter, as in 4.2. Many also fail the **call-rate** filter: mostly the same rare alleles of repeats, which are hard to genotype. A row can fail more than one filter, so the numbers add up to more than the rows removed. (`~` means "not" in Hail, like `!` in R.)

In [ ]:
mt = mt.filter_rows(passes_maf & passes_call & passes_hwe)
mt.count_rows()   # rows we'll test

**🔍 Read the output:** the number of rows left to test. Splitting matters here: in 4.2, PLINK checked a multi-allelic variant's frequency as a whole; here each ALT allele must pass **on its own**. So many rare ALT alleles of repeats drop out, while a common one can stay.

## Part 5. Run the GWAS

`hl.logistic_regression_rows()` runs one logistic regression per row, as `plink2 --glm` did:

| Argument | Means |
|---|---|
| `test="wald"` | the standard Wald test |
| `y=mt.pheno.is_case` | the outcome: case or control |
| `x=mt.GT.n_alt_alleles()` | the genotype as **0, 1, or 2** copies of the ALT allele |
| `covariates=[1.0, ...]` | the covariates. `1.0` is the **intercept**: Hail needs it written out (R and PLINK add it for you) |

Two small differences from PLINK:

- **Missing genotypes:** Hail fills them in with the variant's average (**mean imputation**); PLINK leaves those people out of that variant's test.
- **Firth:** in 4.2 we asked PLINK to switch to Firth regression when the usual test fails; here we use the Wald test throughout. After QC, this rarely matters.

It runs in seconds on the small copy.

In [ ]:
covariates = [1.0, mt.pheno.age, mt.pheno.sex_male] + [mt.pheno[pc] for pc in PCS]

gwas = hl.logistic_regression_rows(
    test="wald",
    y=mt.pheno.is_case,
    x=mt.GT.n_alt_alleles(),
    covariates=covariates,
)
gwas.describe()

**🔍 Read the output:** the result is a Hail **Table**, one row per variant, keyed by `locus` and `alleles`, with:

| Field | Means |
|---|---|
| `beta` | the log odds ratio per ALT allele: **OR = e^beta** |
| `standard_error`, `z_stat` | its standard error and test statistic |
| `p_value` | the p-value |
| `fit` | details of the model fit (e.g. whether it converged) |

## Part 6. Results and plots

### From Hail to pandas

The results are small (one row per variant, no individual data), so we bring them into **pandas** with `to_pandas()`. First we flatten them into simple columns: `key_by()` with nothing removes the keys, and `select()` picks the fields, like `transmute()` in R.

In [ ]:
flat = gwas.key_by()   # the same table, without keys
results = flat.select(
    POS = flat.locus.position,
    REF = flat.alleles[0],     # [0] is the first item: Python counts from 0
    ALT = flat.alleles[1],
    beta = flat.beta,
    p_value = flat.p_value,
).to_pandas()

results = results.dropna(subset=["p_value"])   # drop tests that didn't work
results["OR"] = np.exp(results["beta"])

N_TESTS          = len(results)
REGION_THRESHOLD = 0.05 / N_TESTS
GENOME_THRESHOLD = 5e-8
print(f"{N_TESTS:,} tests; region threshold {REGION_THRESHOLD:.1e}")

# The top 10, with long alleles shortened for display
(results.sort_values("p_value")
        .head(10)
        .assign(REF=lambda d: d["REF"].str.slice(0, 12), ALT=lambda d: d["ALT"].str.slice(0, 12),
                OR=lambda d: d["OR"].round(2),
                p_value=lambda d: d["p_value"].map("{:.1e}".format))   # e.g. 2.4e-08
        [["POS", "REF", "ALT", "OR", "p_value"]])

**🔍 Read the output:** compare with the top 10 from 4.2. The **same part of *TCF7L2*** should be at the top, with similar ORs (about 1.3). Some rows may be different ALT alleles of the same repeat, at the same position.

pandas method chains read top to bottom like an R pipe: `sort_values()` is `arrange()`, `head()` is `head()`, `assign()` is `mutate()`, and `[[...]]` is `select()`.

### rs7903146

In [ ]:
pd.set_option("display.float_format", "{:.3g}".format)   # show small numbers as e.g. 3.3e-06, not 0.000003

results[results["POS"] == LEAD_POS][["POS", "REF", "ALT", "OR", "p_value"]]

**🔍 Read the output:** the **C → T** row is rs7903146's risk allele. In 4.2, PLINK's adjusted OR was about **1.25**: Hail's should be very close. Splitting means the rare **C → G** allele has its own row here, if it passed QC (it almost certainly didn't).

### Regional plot

The same kind of plot as in 4.2, with **matplotlib**, Python's main plotting library. (We skip the LD colours this time.)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(results["POS"] / 1e6, -np.log10(results["p_value"]), s=8, color="#3182bd")
lead = results[(results["POS"] == LEAD_POS) & (results["ALT"] == "T")]
ax.scatter(lead["POS"] / 1e6, -np.log10(lead["p_value"]), s=80, marker="D",
           color="#d7301f", edgecolor="black", label="rs7903146", zorder=3)

ax.axhline(-np.log10(GENOME_THRESHOLD), linestyle="--", color="black", label="genome-wide threshold")
ax.axhline(-np.log10(REGION_THRESHOLD), linestyle=":", color="black", label="region threshold")
ax.plot([GENE_START / 1e6, min(GENE_END, 113100000) / 1e6], [-0.4, -0.4], linewidth=4, color="0.3")
ax.text(GENE_START / 1e6, -1.0, "TCF7L2 →", style="italic")

ax.set_xlabel("Position on chromosome 10 (millions, GRCh38)")
ax.set_ylabel("-log10(p)")
ax.set_ylim(bottom=-1.4)   # room for the gene label
ax.set_title("Type 2 diabetes association, Hail")
ax.legend(loc="upper left")
plt.show()

**🔍 Read the output:** the peak should be in the same place as in 4.2. The gene bar runs off the right edge because *TCF7L2* continues past our window.

**💬 Discuss:** in our test run the plot showed **two** clusters of small p-values: one around rs7903146 (113.00 million) and one a little to the right (113.02 to 113.05), with the top hit. Is that **one** signal or **two**? Variants in LD carry the same signal, but LD in a mixed-ancestry sample is patchy. One way to find out is a **conditional analysis**: add rs7903146's genotype as a covariate and see whether the second cluster stays significant. If it disappears, it was the same signal.

### ✍️ Your turn

How many rows pass the **region** threshold, and how many the **genome-wide** threshold?

<details>
<summary><b>Show answer</b></summary>

```python
print((results["p_value"] < REGION_THRESHOLD).sum(), "pass the region threshold")
print((results["p_value"] < GENOME_THRESHOLD).sum(), "pass the genome-wide threshold")
```

`(...)` gives True/False for each row, and `.sum()` counts the Trues, like `sum()` in R. The region threshold is a little stricter than in 4.2, because splitting gives more rows to test.

</details>

In [ ]:
# Your code here

## Part 7. Compare with PLINK

We load PLINK's results file from 4.2 (`gwas_adjusted.t2d.glm.logistic.hybrid`, in the bucket's `genomics` folder: pandas can read it as tab-separated text) and **match** variants on position, REF, and ALT allele. In PLINK's table the tested allele is `A1`; in Hail's, it's `ALT`.

Not every variant matches:

- **Different windows:** PLINK tested 700,000 letters; we keep only the 200,000 here.
- **Multi-allelic variants:** PLINK's `REF`/`ALT` list every allele (`T,G`); after splitting, Hail's `REF`/`ALT` are one pair, sometimes **trimmed** to their shortest form for repeats. Single-letter variants (SNPs) match well; repeats often don't.
- **QC:** the frequency filter works per allele in Hail, per variant in PLINK.

`pd.merge(..., how="inner")` keeps only variants in **both**, like `inner_join()` in R.

In [ ]:
# PLINK's own results file from 4.2 (tab-separated)
plink_file = glob.glob(os.path.join(DATA_DIR, "genomics", "gwas_adjusted.*glm*"))[0]
plink = pd.read_csv(plink_file, sep="\t", dtype={"REF": str, "A1": str})
plink = plink[["POS", "REF", "A1", "OR", "P"]].rename(columns={"A1": "ALT", "OR": "OR_plink", "P": "p_plink"})

both = pd.merge(results.rename(columns={"OR": "OR_hail", "p_value": "p_hail"}),
                plink, on=["POS", "REF", "ALT"], how="inner")

print(f"{len(results):,} Hail tests, {len(both):,} matched to PLINK")
print("Correlation of log(OR):", round(np.corrcoef(np.log(both["OR_hail"]), np.log(both["OR_plink"]))[0, 1], 4))
print("Correlation of -log10(p):", round(np.corrcoef(-np.log10(both["p_hail"]), -np.log10(both["p_plink"]))[0, 1], 4))

both[both["POS"] == LEAD_POS][["POS", "REF", "ALT", "OR_hail", "OR_plink", "p_hail", "p_plink"]]

**🔍 Read the output:**

- The **correlations** should be very close to **1**: for the variants both tools tested, they agree.
- rs7903146's OR and p-value should be nearly identical in both. Small differences come from missing genotypes (mean imputation vs leaving people out) and Firth vs Wald.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].scatter(both["OR_plink"], both["OR_hail"], s=8, color="#3182bd")
axes[0].set(xlabel="OR, PLINK", ylabel="OR, Hail", title="Odds ratios")

axes[1].scatter(-np.log10(both["p_plink"]), -np.log10(both["p_hail"]), s=8, color="#3182bd")
axes[1].set(xlabel="-log10(p), PLINK", ylabel="-log10(p), Hail", title="p-values")

for ax in axes:
    lo, hi = ax.get_xlim()
    ax.plot([lo, hi], [lo, hi], color="0.5", linewidth=1)   # the line where both agree

plt.tight_layout()
plt.show()

**🔍 Read the output:** points on the grey line mean the two tools agree. A few points off the line are usually variants with missing genotypes, or where PLINK used Firth regression.

**💬 Discuss:** two tools, the same data, the same model: why does it matter that the results agree? (Reproducibility: a result that changes with the software is a warning sign. Checking a key finding with a second tool is good practice.)

### ✍️ Your turn

Run the GWAS again with `test="firth"` instead of `"wald"`, and compare rs7903146's p-value. Does it change?

<details>
<summary><b>Show answer</b></summary>

```python
firth = hl.logistic_regression_rows(test="firth", y=mt.pheno.is_case,
                                    x=mt.GT.n_alt_alleles(), covariates=covariates)
firth = firth.filter(firth.locus.position == LEAD_POS)
firth.select(OR = hl.exp(firth.beta), p_value = firth.p_value).show()
```

Almost no change: Firth regression matters for **rare** alleles or very unbalanced groups, and rs7903146's T allele is common. `.show()` prints a few rows of a Hail Table. Firth is slower, so on a whole genome you'd use it only where needed, as PLINK's `firth-fallback` does.

</details>

In [ ]:
# Your code here

## Save the results

Summary statistics only: one row per variant, no individual data.

In [ ]:
results.to_csv(os.path.join(DATA_DIR, "gwas_tcf7l2_hail.csv"), index=False)
print("Saved gwas_tcf7l2_hail.csv to", DATA_DIR)

## Scaling up

Today, Hail read 200,000 letters for 535,000 people in about 2 to 3 minutes on one app. A genome-wide analysis reads about **15,000 times** more, so it needs a **cluster**: many computers sharing the work, which is what Spark is built for. The same Hail code runs on a cluster unchanged.

- Before running anything genome-wide, time one chromosome (chromosome 22 is the smallest) and multiply, to estimate the cost.
- Clusters cost money **while they run**, even when idle: set them to shut down automatically.
- *All of Us* has already run genome-wide analyses of thousands of phenotypes with Hail: the **All by All** results, which we use in **4.4**.

👩‍🏫 **Instructor tip:** if students' checkpoints take much longer than 2 minutes, the app may be small. Our test app ran the 100,000-letter test in about a minute. Hail's progress bar (`Stage … (6 + 1) / 7`) shows how many pieces are left.

## ⚠️ Common errors

| Message | Means |
|---|---|
| *ModuleNotFoundError: No module named 'hail'* | The notebook is using the wrong kernel, or not an *All of Us* app. Choose **Python 3** (top right) |
| *No module named 'aou_setup'* | The notebook isn't in `notebooks/omics/`, so `..` doesn't point to `notebooks/` |
| *FileNotFoundError … gwas_samples.csv* | Run **4.1** first |
| *IndexError: list index out of range* when loading PLINK's results | PLINK's results file isn't there: run **4.2** through Part 4 first |
| *Java heap space* / *OutOfMemoryError* | The app ran out of memory. Use a smaller window, or a bigger app |
| *HailException: … key type mismatch* | The sample IDs aren't text. Check `dtype={"person_id": str}` |
| A cell runs for many minutes | Lazy evaluation: it triggered reading from the bucket. Make sure you're working from the checkpoint |
| An error reading `~/hail_checkpoints/…` | The checkpoint was interrupted part-way. Set `REBUILD = True` and rerun that cell |

## ✅ Key takeaways

- **Hail** works with genomes as a **MatrixTable**: variants × people, with genotypes as entries
- Hail is **lazy**: it plans until you ask for a result; **checkpoint** after the expensive step
- `splitMT` gives **one row per ALT allele**, so counts and QC differ slightly from PLINK
- The same model in **PLINK** and **Hail** gives the same answers for the same variants: a good reproducibility check
- Hail scales to the whole genome on a **cluster**; for one region, a standard app is enough

**Next:** **4.4 phewas**: flip the question. Instead of many variants and one disease, take **one variant** (rs7903146) and test it against **many** diseases.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).